# DokuAgent: fine-tuning Qwen2.5-1.5B (QLoRA) untuk ekstraksi invoice

Jalankan sel satu per satu dari atas ke bawah (Shift + Enter).

**Sebelum mulai:** menu *Runtime → Change runtime type → T4 GPU → Save*.

## 1. Cek GPU
Harus muncul `Tesla T4`. Kalau error, GPU belum aktif.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install Unsloth (±3–5 menit)
Kalau muncul tombol *Restart session*, klik, lalu lanjut ke sel berikutnya (tidak perlu install ulang).

In [ ]:
%%capture
!pip install unsloth

## 3. Upload data
Pilih **train.jsonl** dan **val.jsonl** dari folder `dokuagent/data/sft/` di laptopmu.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
for f in ['train.jsonl', 'val.jsonl']:
    assert os.path.exists(f), f'{f} belum diupload!'
    print(f, sum(1 for _ in open(f)), 'baris')   # harus 480 dan 60

## 4. Load model 4-bit + pasang LoRA

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

## 5. Siapkan dataset
Cek: teks yang tercetak harus memuat `<|im_start|>assistant`, dan panjang token maksimum harus di bawah 2048.

In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files={'train': 'train.jsonl', 'val': 'val.jsonl'})
def fmt(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False)}
ds = ds.map(fmt, remove_columns=['messages', 'id'])

lens = [len(tokenizer(t)['input_ids']) for t in ds['train']['text']]
print('token terpanjang:', max(lens), '| rata-rata:', sum(lens) // len(lens))
print(ds['train'][0]['text'][-700:])

## 6. Training (±10–20 menit)
Perhatikan kolom *Training Loss*: harus turun. Eval loss dicetak di akhir tiap epoch.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

args = SFTConfig(
    dataset_text_field='text',
    max_seq_length=2048,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=5,
    lr_scheduler_type='linear',
    optim='adamw_8bit',
    weight_decay=0.01,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    eval_strategy='epoch',
    output_dir='outputs',
    report_to='none',
    seed=3407,
)
kw = dict(model=model, train_dataset=ds['train'], eval_dataset=ds['val'], args=args)
try:
    trainer = SFTTrainer(tokenizer=tokenizer, **kw)
except TypeError:   # versi TRL baru memakai nama argumen processing_class
    trainer = SFTTrainer(processing_class=tokenizer, **kw)

# loss hanya dihitung di jawaban (JSON), bukan di teks OCR
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)
stats = trainer.train()

## 7. Grafik loss (screenshot untuk README)

In [ ]:
import matplotlib.pyplot as plt

h = trainer.state.log_history
tr = [(x['step'], x['loss']) for x in h if 'loss' in x]
ev = [(x['step'], x['eval_loss']) for x in h if 'eval_loss' in x]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label='train loss')
if ev: plt.plot(*zip(*ev), 'o-', label='val loss')
plt.xlabel('step'); plt.ylabel('loss'); plt.title('DokuAgent QLoRA'); plt.legend(); plt.grid(alpha=.3)
plt.savefig('loss.png', dpi=150, bbox_inches='tight'); plt.show()
print('eval loss per epoch:', [round(l, 4) for _, l in ev])

## 8. Uji cepat di 3 invoice validasi
Bandingkan PREDIKSI dengan LABEL. Idealnya sama persis.

In [ ]:
import json
FastLanguageModel.for_inference(model)

rows = [json.loads(l) for l in open('val.jsonl')][:3]
for ex in rows:
    msgs = ex['messages'][:2]   # system + user saja
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                           return_tensors='pt').to('cuda')
    out = model.generate(input_ids=inputs, max_new_tokens=1024, do_sample=False)
    pred = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    gold = ex['messages'][2]['content']
    print('==', ex['id'], '| SAMA PERSIS' if json.loads(pred) == json.loads(gold) else '| BEDA')
    print('PREDIKSI:', pred)
    print('LABEL   :', gold, '\n')

## 9. Simpan adapter + ekspor GGUF (±5–10 menit)

In [ ]:
model.save_pretrained('dokuagent-lora')
tokenizer.save_pretrained('dokuagent-lora')
model.save_pretrained_gguf('dokuagent-gguf', tokenizer, quantization_method='q4_k_m')
!ls -lh dokuagent-gguf

## 10. Download ke laptop
File `.gguf` (±1 GB) dan `loss.png` akan terunduh. Kalau browser memblokir, buka panel **Files** di kiri, klik titik tiga pada file, lalu *Download*.

In [ ]:
import glob
gguf = sorted(glob.glob('dokuagent-gguf/**/*.gguf', recursive=True) + glob.glob('*.gguf'))
print(gguf)
q4 = [g for g in gguf if 'q4_k_m' in g.lower()] or gguf
files.download(q4[0])
files.download('loss.png')

## 11. (Opsional, disarankan) Upload ke Hugging Face Hub
Buat token *write* di huggingface.co/settings/tokens, lalu ganti `USERNAME`.

In [ ]:
from huggingface_hub import login
login()

HF_USER = 'USERNAME'
model.push_to_hub(f'{HF_USER}/dokuagent-qwen2.5-1.5b-lora')
tokenizer.push_to_hub(f'{HF_USER}/dokuagent-qwen2.5-1.5b-lora')
model.push_to_hub_gguf(f'{HF_USER}/dokuagent-qwen2.5-1.5b-gguf', tokenizer,
                       quantization_method='q4_k_m')